# BrAPI MCP Server — Python client walkthrough

This notebook drives **[@cyanheads/brapi-mcp-server](https://github.com/cyanheads/brapi-mcp-server)** — a real, running
[MCP](https://modelcontextprotocol.io/) server for [BrAPI](https://brapi.org/) (Breeding API) data — from Python, using
the official [`mcp`](https://pypi.org/project/mcp/) client SDK.

It's a Python port of an earlier hackathon demo that showed an agent routing natural-language prompts
(*"show me available studies and summarize them"*) to BrAPI tools, backed by a small mocked local library since no
Python client examples existed for this server. This version talks to the real server over stdio and uses its actual
tools — including the server-side SQL analysis layer (DuckDB-backed `brapi_dataframe_query`) — instead of a mock.

**What you'll see:**
1. Launch the server as a subprocess, list its tools, and `brapi_connect` to a BrAPI v2 server.
2. `brapi_find_studies` — results beyond a small row cap spill into a queryable in-memory dataframe.
3. `brapi_dataframe_describe` / `brapi_dataframe_query` — count, filter, and group by with plain SQL.

**Prerequisites:** Python 3.10+, `pip install mcp`, and Node.js or Bun on `PATH` (used to launch the server via `npx`/`bunx`).

**Note on structure:** every MCP call below runs inside one `run_demo()` coroutine, invoked from a single cell, rather
than spread across cells that each hold the session open. `stdio_client`'s connection is an `anyio` task-group-based
context manager, and anyio ties a task group to the asyncio task that entered it — opening it in one Jupyter cell and
closing it in another means closing from a *different* task, which reliably hangs. Keeping the whole session lifecycle
inside one coroutine/task sidesteps that; the cells below just print pieces of what it already collected.

In [ ]:
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Launches `npx -y @cyanheads/brapi-mcp-server@latest` as a subprocess and speaks MCP to it over stdio.
# Swap "npx" for "bunx" if you prefer Bun.
SERVER_PARAMS = StdioServerParameters(
    command="npx",
    args=["-y", "@cyanheads/brapi-mcp-server@latest"],
    env={"MCP_TRANSPORT_TYPE": "stdio", "MCP_LOG_LEVEL": "error"},
)


async def run_demo() -> dict:
    """Connect, find studies, and run SQL analysis -- all within one session/task."""
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()

            connect = await session.call_tool(
                "brapi_connect",
                {"baseUrl": "https://test-server.brapi.org/brapi/v2", "alias": "demo"},
            )
            envelope = connect.structured_content

            # loadLimit=1 forces spillover to a dataframe on this tiny 3-row test dataset.
            found = await session.call_tool("brapi_find_studies", {"alias": "demo", "loadLimit": 1})
            studies = found.structured_content
            table = studies["dataframe"]["tableName"]

            described = await session.call_tool("brapi_dataframe_describe", {"dataframe": table})
            columns = described.structured_content["tables"][0]["columns"]

            grouped = await session.call_tool(
                "brapi_dataframe_query",
                {"sql": f"SELECT studyType, COUNT(*) AS n FROM {table} GROUP BY studyType"},
            )
            filtered = await session.call_tool(
                "brapi_dataframe_query",
                {"sql": f"SELECT studyDbId, studyName FROM {table} WHERE studyName ILIKE '%yield%'"},
            )
            counted = await session.call_tool(
                "brapi_dataframe_query",
                {"sql": f"SELECT COUNT(*) AS study_count FROM {table}"},
            )

            return {
                "tool_count": len(tools.tools),
                "sample_tools": [t.name for t in tools.tools if "find" in t.name or "dataframe" in t.name],
                "server_name": envelope["server"]["name"],
                "capabilities_count": len(envelope["capabilities"]),
                "total_studies": studies["totalCount"],
                "dataframe": studies["dataframe"],
                "columns": [c["name"] for c in columns],
                "group_by_rows": grouped.structured_content["rows"],
                "filter_rows": filtered.structured_content["rows"],
                "count_rows": counted.structured_content["rows"],
            }


results = await run_demo()
print(f"Connected — {results['tool_count']} tools available")
results["sample_tools"]

## Connect to a BrAPI server

`brapi_connect` registers a server under an alias and returns an orientation envelope (identity, capabilities,
attribution). We pointed it at BrAPI's public dummy-data test server above so this notebook runs without credentials;
swap in one of the server's [built-in aliases](https://github.com/cyanheads/brapi-mcp-server#built-in-aliases)
(e.g. `bti-cassava` for live NextGen Cassava data on cassavabase.org) to try it against real breeding data — those
are third-party servers, so allow for occasional upstream latency/timeouts that aren't this notebook's fault.

In [ ]:
print("Connected to:", results["server_name"])
print("Capabilities advertised:", results["capabilities_count"])

## Find studies — with automatic dataframe spillover

`brapi_find_studies` returns a page of rows inline, plus (once the result exceeds `loadLimit`) a `dataframe` handle
naming an in-memory table you can query with SQL — no re-fetching, no client-side pagination logic.

In [ ]:
df = results["dataframe"]
print(f"Found {results['total_studies']} studies -> staged as dataframe `{df['tableName']}` ({df['rowCount']} rows)")
print(f"{len(results['columns'])} columns, e.g.:", results["columns"][:6])

## Analyze with SQL — count, filter, group by

This is the piece the original hackathon MCP didn't have: no way to filter, count, or aggregate server-side, so
every such operation had to happen in client code after downloading the raw data. `brapi_dataframe_query` runs
read-only SQL (DuckDB) directly against the staged rows.

In [ ]:
print("group_by_counts equivalent (SQL GROUP BY):")
for row in results["group_by_rows"]:
    print(" ", row)

In [ ]:
print("filter_results equivalent (SQL WHERE ILIKE):")
for row in results["filter_rows"]:
    print(" ", row)

In [ ]:
print("count_results equivalent (SQL COUNT):", results["count_rows"])